# 5a) Core-Ring Map Functions

Shared functions for constructing coordination-core ligand descriptors and generating interactive core-ring maps.

Execution notebooks should `%run ./5a-Core_Ring_Map_Functions.ipynb` and then provide:

- `systems`: list of SCOPE `System` objects to map.
- `data`: dictionary keyed by refcode with values to show in tooltips and/or use for color.
- `properties_to_show`: dictionary defining which properties appear in hover boxes.
- `color_property`: property key used for point colors.
- `target_denticity`: denticity block to map.
- `embedding_method`: `"mds"` or `"pca"`.
- `grid`: `True` for sparse grid layout, `False` for continuous tree layout.

In [1]:
import csv
import json
import re
from collections import deque
from html import escape
from pathlib import Path

import numpy as np
import pandas as pd
import scope
from IPython.display import IFrame, SVG, display
from matplotlib import colormaps
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize, to_hex
from matplotlib.ticker import MaxNLocator
from rdkit import Chem, RDLogger
from rdkit.Chem import Draw, rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold
from scope.operations.graphs import compute_topological_distances
from sklearn.decomposition import PCA
from sklearn.manifold import smacof
from sklearn.metrics import pairwise_distances

RDLogger.DisableLog('rdApp.warning')


## CSV-related

In [ ]:
def read_csv(path):
    with open(path, newline='') as f:
        return list(csv.DictReader(f))

def count_csv_entries(csv_path, column, value):
    count = 0
    with open(csv_path, newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row[column] == value:
                count += 1
    return count

def get_csv_entries(csv_path, column, value):
    matches = []
    with open(csv_path, newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row[column] == value:
                matches.append(row)
    return matches 

# rows = filter_rows("rt_systems.csv", lambda row: float(row["t12"]) < 300)
def filter_rows(csv_path, condition, first: bool=False):
    matches = []
    with open(csv_path, newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if condition(row):
                matches.append(row)
                if first: return row
    return matches

def sum_csv_entries(csv_path, column):
    total = 0
    with open(csv_path, newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            value = row[column].strip()
            if value:
                total += int(value)
    return total

def get_values(csv_path, column):
    values = []
    with open(csv_path, newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            values.append(float(row[column]))
    return values

## Other

In [3]:
def resolve_system_path(base_path, specific_path=None):
    path = Path(specific_path)
    if path.is_absolute() and path.exists():
        return path
    return Path(base_path) / str(specific_path).lstrip('/')

def as_float(value):
    if value in (None, '', 'None'):
        return None
    return float(value)

def get_last_author(authors):
    author_parts = [part.strip() for part in str(authors).split(';') if part.strip()]
    return author_parts[-1] if author_parts else 'unknown'

## Descriptor Construction

In [4]:
def display_svg_mol(mol, legend=None):
    svg = Draw.MolsToGridImage([mol], molsPerRow=1, subImgSize=(520, 360), legends=[legend], useSVG=True)
    display(SVG(svg.data if hasattr(svg, 'data') else str(svg)))

def color_atom_labels(svg_text, atom_indices, color='#D62728'):
    for atom_idx in atom_indices:
        pattern = rf'(<path class="atom-{int(atom_idx)}"[^>]*fill=")#[0-9A-Fa-f]{{6,8}}'
        svg_text = re.sub(pattern, rf'\1{color}', svg_text)
    return svg_text

def display_svg_mol_extra(record, size=(620, 420)):
    highlight_atoms = sorted(set(record['core_atoms']) | set(record['subs_atoms']))
    highlight_colors = {idx: (0.65, 0.90, 0.70) for idx in record['core_atoms']}
    highlight_colors.update({idx: (0.70, 0.82, 1.00) for idx in record['subs_atoms']})
    donor_atoms = {
        int(record['core_atoms'][atom.GetIdx()])
        for atom in record['core_mol'].GetAtoms()
        if atom.HasProp('_MetalDonor') and atom.GetBoolProp('_MetalDonor')
    }
    svg = Draw.MolsToGridImage(
        [record['lig_mol']],
        molsPerRow=1,
        subImgSize=size,
        legends=[f"{record['id']} | green=core, blue=substituent, red=coordinating N labels"],
        highlightAtomLists=[highlight_atoms],
        highlightAtomColors=[highlight_colors],
        useSVG=True,
    )
    svg_text = svg.data if hasattr(svg, 'data') else str(svg)
    display(SVG(color_atom_labels(svg_text, donor_atoms)))

def get_hs_ref_mol(system):
    found, molecule = system.find_source('ref_hs_mol')
    if not found: raise ValueError(f'System {system.refcode} is missing ref_hs_mol')
    return molecule

def get_rdkit_info(ligand):
    if hasattr(ligand, 'rdkit_obj') and ligand.rdkit_obj is not None:
        rdkit_mol = Chem.Mol(ligand.rdkit_obj)
    elif hasattr(ligand, 'smiles') and ligand.smiles:
        rdkit_mol = Chem.MolFromSmiles(ligand.smiles)
    else:
        raise ValueError('Could not build an RDKit object for ligand')

    if hasattr(ligand, 'smiles') and ligand.smiles:
        smiles = ligand.smiles
    else:
        smiles = Chem.MolToSmiles(rdkit_mol)
    return rdkit_mol, smiles

def morgan_fp_array(mol, radius=2, n_bits=1024):
    from rdkit import DataStructs
    fp = rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=int)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr


### Version 1 (Maewenn's work)

In [5]:
def get_ligand_descriptor_type1(core_mol, subs_mol, radius=2, n_bits=1024):
    core_fp         = morgan_fp_array(core_mol, radius=radius, n_bits=n_bits)
    substituent_fp  = morgan_fp_array(subs_mol, 0)
    return {'core_fp':core_fp, 'substituent_fp':substituent_fp}

### Version 2: Distance-Weighted Morgan Descriptor

#### Ligand splitting into coordination core and substituent


In [6]:
def molecule_graph_with_current_metal_connectivity(molecule, debug: int=0):
    molecule_graph = molecule.get_graph(debug=debug).copy()
    metal_indices = {metal.get_parent_index('molecule') for metal in molecule.metals}

    # Bond objects can retain metal contacts inconsistent with the current madjmat.
    molecule_graph.remove_edges_from([
        (first, second)
        for first, second in molecule_graph.edges()
        if first in metal_indices or second in metal_indices
    ])
    for metal_index in metal_indices:
        for atom_index, connected in enumerate(molecule.madjmat[metal_index]):
            if connected and atom_index != metal_index:
                distance = np.linalg.norm(
                    np.asarray(molecule.atoms[metal_index].coord, dtype=float)
                    - np.asarray(molecule.atoms[atom_index].coord, dtype=float)
                )
                molecule_graph.add_edge(metal_index, atom_index, distance=float(distance))
    return molecule_graph

def get_metal_bound_ligand_atoms(ligand):
    ligand_indices = ligand.get_parent_indices('molecule')
    molecule       = ligand.get_parent('molecule')
    metal_index    = molecule.metals[0].get_parent_index('molecule')
    return {
        ligand_index
        for ligand_index, molecule_index in enumerate(ligand_indices)
        if molecule.madjmat[metal_index, molecule_index] > 0
    }

def metal_connected_ring_core_atoms(mol, metal_bound_atoms):
    if not metal_bound_atoms:
        return set()

    rings = [set(ring) for ring in mol.GetRingInfo().AtomRings()]
    selected_ring_atoms = set()
    seen = set()

    for start in range(len(rings)):
        if start in seen:
            continue

        stack = [start]
        ring_system_atoms = set()

        while stack:
            ring_id = stack.pop()
            if ring_id in seen:
                continue

            seen.add(ring_id)
            ring_system_atoms.update(rings[ring_id])

            for other_id, other_ring in enumerate(rings):
                if other_id not in seen and rings[ring_id] & other_ring:
                    stack.append(other_id)

        if ring_system_atoms & metal_bound_atoms:
            selected_ring_atoms.update(ring_system_atoms)

    return selected_ring_atoms

def donor_backbone_core_atoms(mol, metal_bound_atoms, monodentate_radius=2):
    donors = sorted(metal_bound_atoms)
    if not donors:
        return set()

    if len(donors) == 1:
        backbone = {donors[0]}
        frontier = {donors[0]}
        for _ in range(monodentate_radius):
            next_frontier = {
                nbr.GetIdx()
                for atom_idx in frontier
                for nbr in mol.GetAtomWithIdx(atom_idx).GetNeighbors()
                if nbr.GetAtomicNum() != 1 and nbr.GetIdx() not in backbone
            }
            backbone.update(next_frontier)
            frontier = next_frontier
        return backbone

    backbone = set(donors)
    for i, first in enumerate(donors):
        for second in donors[i + 1:]:
            backbone.update(Chem.GetShortestPath(mol, first, second))
    return {idx for idx in backbone if mol.GetAtomWithIdx(idx).GetAtomicNum() != 1}

def mol_fragment_from_atom_indices(mol, atom_indices):
    atom_indices = sorted(set(int(idx) for idx in atom_indices))
    if not atom_indices:
        return Chem.MolFromSmiles('')

    index_map = {}
    fragment = Chem.RWMol()
    for old_idx in atom_indices:
        atom = Chem.Atom(mol.GetAtomWithIdx(old_idx))
        # Prevent RDKit from saturating cut bonds with new implicit hydrogens.
        atom.SetNoImplicit(True)
        index_map[old_idx] = fragment.AddAtom(atom)

    atom_set = set(atom_indices)
    for bond in mol.GetBonds():
        begin = bond.GetBeginAtomIdx()
        end = bond.GetEndAtomIdx()
        if begin in atom_set and end in atom_set:
            fragment.AddBond(index_map[begin], index_map[end], bond.GetBondType())
            new_bond = fragment.GetBondBetweenAtoms(index_map[begin], index_map[end])
            new_bond.SetIsAromatic(bond.GetIsAromatic())

    fragment = fragment.GetMol()
    fragment.UpdatePropertyCache(strict=False)
    Chem.GetSymmSSSR(fragment)
    return fragment

def fragment_composition_label(fragment):
    counts = {}
    for atom in fragment.GetAtoms():
        if atom.GetAtomicNum() == 1:
            continue
        symbol = atom.GetSymbol()
        counts[symbol] = counts.get(symbol, 0) + 1

    return ''.join(f'{symbol}{counts[symbol] if counts[symbol] > 1 else ""}' for symbol in sorted(counts)) or 'H'

def topology_graph_label(mol):
    heavy_atoms = [atom.GetIdx() for atom in mol.GetAtoms() if atom.GetAtomicNum() != 1]
    index_map = {}
    topology = Chem.RWMol()

    for old_idx in heavy_atoms:
        atom = Chem.Atom(mol.GetAtomWithIdx(old_idx).GetAtomicNum())
        atom.SetFormalCharge(0)
        atom.SetIsAromatic(False)
        atom.SetNoImplicit(True)
        index_map[old_idx] = topology.AddAtom(atom)

    heavy_atom_set = set(heavy_atoms)
    for bond in mol.GetBonds():
        begin = bond.GetBeginAtomIdx()
        end = bond.GetEndAtomIdx()
        if begin in heavy_atom_set and end in heavy_atom_set:
            topology.AddBond(index_map[begin], index_map[end], Chem.BondType.SINGLE)

    topology = topology.GetMol()
    topology.UpdatePropertyCache(strict=False)
    return Chem.MolToSmiles(topology, canonical=True)

def fragment_family_label(mol):
    fragments = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=False)
    labels = sorted(fragment_composition_label(fragment) for fragment in fragments)
    compact_labels = []
    for label in labels:
        if compact_labels and compact_labels[-1][0] == label:
            compact_labels[-1][1] += 1
        else:
            compact_labels.append([label, 1])
    return '.'.join(
        f'{count}x[{label}]' if count > 1 else f'1x[{label}]'
        for label, count in compact_labels
    )

def donor_degree_label(core_mol, non_ring_only=False):
    ring_atoms = {idx for ring in core_mol.GetRingInfo().AtomRings() for idx in ring}
    degrees = []
    for atom in core_mol.GetAtoms():
        if not atom.HasProp('_MetalDonor') or not atom.GetBoolProp('_MetalDonor'):
            continue
        if non_ring_only and atom.GetIdx() in ring_atoms:
            continue
        degrees.append(atom.GetIntProp('_LigandEffectiveDegree'))
    return '.'.join(f'N{degree}' for degree in sorted(degrees))

def hybrid_ring_site_label(core_mol):
    def restricted_distance(allowed_atoms, starts, target):
        queue = deque((start, 0) for start in starts)
        seen = set(starts)
        while queue:
            atom_idx, distance = queue.popleft()
            if atom_idx == target:
                return distance
            for neighbour in core_mol.GetAtomWithIdx(atom_idx).GetNeighbors():
                neighbour_idx = neighbour.GetIdx()
                if neighbour_idx in allowed_atoms and neighbour_idx not in seen:
                    seen.add(neighbour_idx)
                    queue.append((neighbour_idx, distance + 1))
        return -1

    rings = [set(ring) for ring in core_mol.GetRingInfo().AtomRings()]
    ring_systems = []
    seen = set()
    for start in range(len(rings)):
        if start in seen:
            continue
        stack = [start]
        atoms = set()
        while stack:
            ring_id = stack.pop()
            if ring_id in seen:
                continue
            seen.add(ring_id)
            atoms.update(rings[ring_id])
            for other_id, other_ring in enumerate(rings):
                if other_id not in seen and rings[ring_id] & other_ring:
                    stack.append(other_id)
        ring_systems.append(atoms)

    labels = []
    for atoms in ring_systems:
        donors = {
            idx for idx in atoms
            if core_mol.GetAtomWithIdx(idx).HasProp('_MetalDonor')
            and core_mol.GetAtomWithIdx(idx).GetBoolProp('_MetalDonor')
        }
        if not donors:
            continue
        attachments = []
        for atom_idx in atoms:
            atom = core_mol.GetAtomWithIdx(atom_idx)
            if any(
                neighbour.GetIdx() not in atoms and neighbour.GetAtomicNum() != 1
                for neighbour in atom.GetNeighbors()
            ):
                attachments.append(restricted_distance(atoms, donors, atom_idx))
        composition = fragment_composition_label(mol_fragment_from_atom_indices(core_mol, atoms))
        labels.append(
            f"{composition}:donors={len(donors)}:distance={','.join(map(str, sorted(attachments)))}"
        )
    return '|'.join(sorted(labels))

def core_family_label(core_mol):
    if core_mol is None or core_mol.GetNumAtoms() == 0:
        return 'non-ring coordination core'

    ring_atoms = {idx for ring in core_mol.GetRingInfo().AtomRings() for idx in ring}
    heavy_atoms = {atom.GetIdx() for atom in core_mol.GetAtoms() if atom.GetAtomicNum() != 1}

    if not ring_atoms:
        return f'acyclic[{topology_graph_label(core_mol)};degrees={donor_degree_label(core_mol)}]'

    if heavy_atoms - ring_atoms:
        ring_mol = mol_fragment_from_atom_indices(core_mol, ring_atoms)
        return (
            f'hybrid[scaffold={fragment_family_label(ring_mol)};'
            f'sites={hybrid_ring_site_label(core_mol)};'
            f'eff. connectivity={donor_degree_label(core_mol, non_ring_only=True)}]'
        )

    return fragment_family_label(core_mol)

def split_core_substituent(ligand, mols: bool=False, debug: int=0):
    lig_mol, smiles = get_rdkit_info(ligand)
    ligand_indices  = ligand.get_parent_indices('molecule')
    molecule        = ligand.get_parent('molecule')
    metal           = molecule.metals[0]
    metal_index     = metal.get_parent_index('molecule')
    molecule_graph  = molecule_graph_with_current_metal_connectivity(molecule, debug=debug)
    top_distances   = compute_topological_distances(molecule_graph, metal_index)
    lig_distances   = np.array([top_distances[idx] for idx in ligand_indices])
    donor_atoms     = get_metal_bound_ligand_atoms(ligand)

    ring_atoms = set(metal_connected_ring_core_atoms(lig_mol, donor_atoms))
    if ring_atoms and donor_atoms <= ring_atoms:
        core_heavy_atoms = ring_atoms
    else:
        backbone_atoms = set(donor_backbone_core_atoms(lig_mol, donor_atoms))
        if ring_atoms:
            backbone_atoms = {
                atom_idx for atom_idx in backbone_atoms
                if not lig_mol.GetAtomWithIdx(atom_idx).IsInRing()
            }
        core_heavy_atoms = ring_atoms | backbone_atoms
    core_h_atoms    = {atom.GetIdx() for atom in lig_mol.GetAtoms() if atom.GetAtomicNum() == 1 and any(nbr.GetIdx() in core_heavy_atoms for nbr in atom.GetNeighbors())}
    core_atoms      = np.array(sorted(core_heavy_atoms | core_h_atoms), dtype=int)
    subs_atoms      = np.array(sorted(set(range(lig_mol.GetNumAtoms())) - set(core_atoms)), dtype=int)

    if mols:
        core_mol = mol_fragment_from_atom_indices(lig_mol, core_atoms)
        core_index_map = {old_idx: new_idx for new_idx, old_idx in enumerate(core_atoms)}
        for atom_idx in donor_atoms & set(core_atoms):
            core_atom = core_mol.GetAtomWithIdx(core_index_map[atom_idx])
            ligand_atom = lig_mol.GetAtomWithIdx(atom_idx)
            core_atom.SetBoolProp('_MetalDonor', True)
            core_atom.SetIntProp(
                '_LigandEffectiveDegree',
                int(ligand_atom.GetTotalDegree() + max(0, -ligand_atom.GetFormalCharge())),
            )
        subs_mol = mol_fragment_from_atom_indices(lig_mol, subs_atoms)
        return core_mol, subs_mol
    else:
        return core_atoms, subs_atoms


#### Distance-weighted fingerprint creation


In [7]:
def get_weights(ligand, alpha=0.25, debug: int=0):
    def distance_weight(distance, alpha=0.25):
        return np.round(float(np.exp(-alpha * (float(distance) - 1.0))), 4)

    ligand_indices  = ligand.get_parent_indices('molecule')
    molecule        = ligand.get_parent('molecule')
    metal           = molecule.metals[0]
    metal_index     = metal.get_parent_index('molecule')
    molecule_graph  = molecule_graph_with_current_metal_connectivity(molecule, debug=debug)

    top_distances   = compute_topological_distances(molecule_graph, metal_index)
    distances       = np.array([top_distances[idx] for idx in ligand_indices])
    weights         = np.array([distance_weight(distance, alpha=alpha) for distance in distances])
    return weights, distances

def atom_centered_weighted_morgan_fp(mol, atom_weights, centers=None, radius=2, n_bits=1024, normalize=True):
    if centers is None:
        centers = range(mol.GetNumAtoms())

    fp = np.zeros(n_bits, dtype=float)
    contribution_rows = []

    for center_idx in centers:
        center_idx = int(center_idx)
        center_weight = float(atom_weights[center_idx])
        bit_info = {}
        rdMolDescriptors.GetMorganFingerprintAsBitVect(
            mol,
            radius,
            nBits=n_bits,
            fromAtoms=[center_idx],
            bitInfo=bit_info,
        )

        for bit_id, environments in bit_info.items():
            contribution = center_weight * len(environments)
            fp[bit_id] += contribution
            contribution_rows.append({
                'center_atom_idx': center_idx,
                'center_symbol': mol.GetAtomWithIdx(center_idx).GetSymbol(),
                'center_weight': round(center_weight, 4),
                'bit_id': int(bit_id),
                'num_environments_for_bit': len(environments),
                'contribution': round(float(contribution), 4),
                'environment_radii': sorted({int(env_radius) for _, env_radius in environments}),
            })

    norm = np.linalg.norm(fp)
    if normalize and norm > 0:
        fp = fp / norm

    return fp, pd.DataFrame(contribution_rows)

def get_ligand_descriptor_type2(ligand, ligand_mol, alpha=0.25, radius=2, n_bits=1024, debug: int=0):
    weights, distances = get_weights(ligand, alpha=alpha, debug=debug)
    if len(weights) != ligand_mol.GetNumAtoms():
        raise ValueError(f'Ligand/RDKit atom mismatch: {len(weights)} weights for {ligand_mol.GetNumAtoms()} atoms')

    core_mol, _                               = split_core_substituent(ligand, mols=True, debug=debug)
    core_atoms, substituent_atoms             = split_core_substituent(ligand, mols=False, debug=debug)
    core_weights                              = weights[core_atoms]
    if metal_connected_ring_core_atoms(ligand_mol, get_metal_bound_ligand_atoms(ligand)):
        core_fp, core_contributions = atom_centered_weighted_morgan_fp(core_mol, core_weights, radius=radius, n_bits=n_bits)
    else:
        core_fp, core_contributions = atom_centered_weighted_morgan_fp(ligand_mol, weights, centers=core_atoms, radius=radius, n_bits=n_bits)
    substituent_fp, substituent_contributions = atom_centered_weighted_morgan_fp(ligand_mol, weights, centers=substituent_atoms, radius=radius, n_bits=n_bits)

    return {'core_fp': core_fp, 'substituent_fp': substituent_fp, 'core_contributions': core_contributions, 'substituent_contributions': substituent_contributions}


### Construction of Descriptors 1 and 2 from Systems

In [8]:
def collect_ligands_data(systems, descriptor_type: int=2, alpha=0.25, radius=2, n_bits=1024):
    records  = []
    failures = []

    for system in systems:
        refcode = getattr(system, 'refcode_wo_digits', None) or getattr(system, 'refcode', None)
        try:
            hs_ref_mol             = get_hs_ref_mol(system)
            lig                    = hs_ref_mol.ligands[0]
            lig_mol, smiles        = get_rdkit_info(lig)
            core_mol, subs_mol     = split_core_substituent(lig, mols=True)
            core_atoms, subs_atoms = split_core_substituent(lig, mols=False)
            family                 = core_family_label(core_mol)
            is_aromatic            = int(any(atom.GetIsAromatic() for atom in lig_mol.GetAtoms()))
            denticity              = lig.get_denticity()

            if descriptor_type == 1:
                desc = get_ligand_descriptor_type1(core_mol, subs_mol, radius=radius, n_bits=n_bits)
            elif descriptor_type == 2:
                desc = get_ligand_descriptor_type2(lig, lig_mol, alpha=alpha, radius=radius, n_bits=n_bits)
            else:
                raise ValueError('descriptor_type must be 1 or 2')

            records.append({
                'id'            : refcode,
                'smiles'        : smiles,
                'denticity'     : denticity,
                'is_aromatic'   : is_aromatic,
                'family'        : family,
                'core_mol'      : core_mol,
                'subs_mol'      : subs_mol,
                'core_atoms'    : core_atoms,
                'subs_atoms'    : subs_atoms,
                'lig_mol'       : lig_mol,
                'core_fp'       : desc['core_fp'],
                'substituent_fp': desc['substituent_fp'],
            })
        except Exception as exc:
            failures.append((refcode, repr(exc)))

    return records, failures


## Map Layout

In [ ]:

def project_map(matrix, method='mds', metric='cosine'):
    matrix = np.asarray(matrix, dtype=float)
    if len(matrix) == 0:
        return np.empty((0, 2)), None
    if len(matrix) == 1 or np.all(np.var(matrix, axis=0) == 0):
        return np.zeros((len(matrix), 2)), None
    if len(matrix) == 2:
        distance = pairwise_distances(matrix, metric=metric)[0, 1]
        distance = 1.0 if distance == 0 else distance
        return np.array([[-0.5 * distance, 0.0], [0.5 * distance, 0.0]]), None
    if method == 'mds':
        distances = pairwise_distances(matrix, metric=metric)
        coords, stress = smacof(distances, metric=True, n_components=2, random_state=7, normalized_stress='auto')
        return coords, {'stress': stress}
    if method == 'pca':
        model = PCA(n_components=2)
        return model.fit_transform(matrix), model
    raise ValueError("embedding_method must be 'mds' or 'pca'")

def spread_points(coords, min_distance=None, strength=0.35, iterations=450, anchor_strength=0.025):
    coords = np.asarray(coords, dtype=float)
    if len(coords) <= 1:
        return coords.copy()

    spread = coords.copy()
    anchor = coords.copy()
    span = max(np.ptp(coords[:, 0]), np.ptp(coords[:, 1]), 1.0)
    if min_distance is None:
        min_distance = 0.12 * span

    rng = np.random.default_rng(7)
    spread += rng.normal(scale=1e-4 * span, size=spread.shape)

    for _ in range(iterations):
        forces = np.zeros_like(spread)
        for i in range(len(spread)):
            delta = spread[i] - spread
            distances = np.linalg.norm(delta, axis=1)
            crowded = (distances > 1e-12) & (distances < min_distance)
            if np.any(crowded):
                unit = delta[crowded] / distances[crowded, None]
                overlap = (min_distance - distances[crowded])[:, None]
                forces[i] += np.sum(unit * overlap, axis=0)
        forces -= anchor_strength * (spread - anchor)
        spread += strength * forces / max(len(spread), 1)

    return spread

def assign_grid_centers(family_rows, raw_centers, grid_cell_spacing=1.0):
    for row, center in zip(family_rows, raw_centers):
        row['raw_family_x'] = center[0]
        row['raw_family_y'] = center[1]

    ordered_families = sorted(family_rows, key=lambda row: (row['raw_family_y'], row['raw_family_x']))
    n_families = len(ordered_families)
    n_cols = int(np.ceil(np.sqrt(n_families * 1.35)))
    n_rows = int(np.ceil(n_families / n_cols))

    grid_sites = []
    for row_idx in range(n_rows):
        for col_idx in range(n_cols):
            x = (col_idx - (n_cols - 1) / 2) * grid_cell_spacing
            y = - (row_idx - (n_rows - 1) / 2) * grid_cell_spacing
            grid_sites.append((x, y, row_idx + 1, col_idx + 1))

    for family_row, (x, y, row_idx, col_idx) in zip(ordered_families, grid_sites):
        family_row['family_x'] = x
        family_row['family_y'] = y
        family_row['grid_row'] = row_idx
        family_row['grid_column'] = col_idx

    return family_rows

def assign_position_family_labels(family_rows, prefix='F'):
    ordered_rows = sorted(
        family_rows,
        key=lambda row: (-float(row['family_y']), float(row['family_x']), row['Core-ring family']),
    )
    for idx, row in enumerate(ordered_rows, start=1):
        row['family_position_index'] = idx
        if row.get('grid_row') is not None and row.get('grid_column') is not None:
            row['family_id'] = f"{prefix}({int(row['grid_row'])},{int(row['grid_column'])})"
        else:
            row['family_id'] = f'{prefix}{idx}'
    return family_rows

def build_core_ring_map_data(
    descriptor_records,
    target_denticity,
    data=None,
    embedding_method='mds',
    grid=False,
    spread_family_centers=True,
    family_min_distance=0.12,
    grid_cell_spacing=1.0,
    grid_member_radius=0.28,
    local_member_min_distance=0.08,
):
    data = data or {}
    selected_records = [record for record in descriptor_records if str(record['denticity']) == str(target_denticity)]
    if not selected_records:
        raise ValueError(f'No homoleptic systems found for denticity {target_denticity}')

    family_to_records = {}
    for record in selected_records:
        family_to_records.setdefault(record['family'], []).append(record)

    family_rows = []
    for family, records in family_to_records.items():
        branch_vectors = np.asarray([record['core_fp'] for record in records], dtype=float)
        if not np.any(branch_vectors):
            branch_vectors = np.asarray([record['substituent_fp'] for record in records], dtype=float)
        family_rows.append({
            'Core-ring family': family,
            'family_count': len(records),
            'branch_vector': np.mean(branch_vectors, axis=0),
        })

    family_rows = sorted(family_rows, key=lambda row: (-row['family_count'], row['Core-ring family']))
    raw_centers, embedding = project_map([row['branch_vector'] for row in family_rows], method=embedding_method)

    if grid:
        family_rows = assign_grid_centers(family_rows, raw_centers, grid_cell_spacing=grid_cell_spacing)
        local_scale = grid_member_radius
    else:
        family_centers = spread_points(
            raw_centers,
            min_distance=family_min_distance * max(np.ptp(raw_centers[:, 0]), np.ptp(raw_centers[:, 1]), 1.0),
        ) if spread_family_centers else raw_centers
        for row, raw_center, center in zip(family_rows, raw_centers, family_centers):
            row['raw_family_x'] = raw_center[0]
            row['raw_family_y'] = raw_center[1]
            row['family_x'] = center[0]
            row['family_y'] = center[1]
            row['grid_row'] = None
            row['grid_column'] = None

        if len(family_centers) > 1:
            center_distances = pairwise_distances(family_centers)
            nonzero_distances = center_distances[center_distances > 1e-12]
            local_scale = 0.18 * np.median(nonzero_distances) if len(nonzero_distances) else 0.35
        else:
            local_scale = 0.35

    family_rows = assign_position_family_labels(family_rows)
    family_rows = sorted(family_rows, key=lambda row: row['family_position_index'])

    map_rows = []
    for family_row in family_rows:
        family = family_row['Core-ring family']
        records = family_to_records[family]
        local_coords, _ = project_map([record['substituent_fp'] for record in records], method=embedding_method)
        max_radius = np.max(np.linalg.norm(local_coords, axis=1)) if len(local_coords) else 0.0
        if max_radius > 0:
            scale = local_scale if grid else local_scale * min(1.4, 0.75 + 0.07 * np.sqrt(len(records)))
            local_coords = local_coords / max_radius * scale
        else:
            scale = local_scale if grid else local_scale

        if local_member_min_distance and len(local_coords) > 1:
            local_coords = spread_points(
                local_coords,
                min_distance=local_member_min_distance * local_scale,
                strength=0.25,
                iterations=250,
                anchor_strength=0.08,
            )
            max_radius = np.max(np.linalg.norm(local_coords, axis=1)) if len(local_coords) else 0.0
            max_allowed_radius = max(scale * 1.15, local_member_min_distance * local_scale)
            if max_radius > max_allowed_radius:
                local_coords = local_coords / max_radius * max_allowed_radius

        for record, local_coord in zip(records, local_coords):
            row = {
                'Refcode': record['id'],
                'family_id': family_row['family_id'],
                'map_x': family_row['family_x'] + local_coord[0],
                'map_y': family_row['family_y'] + local_coord[1],
                'family_x': family_row['family_x'],
                'family_y': family_row['family_y'],
                'raw_family_x': family_row['raw_family_x'],
                'raw_family_y': family_row['raw_family_y'],
                'local_x': local_coord[0],
                'local_y': local_coord[1],
                'denticity': record['denticity'],
                'is_aromatic': bool(record['is_aromatic']),
                'family_position_index': family_row['family_position_index'],
                'grid_row': family_row.get('grid_row'),
                'grid_column': family_row.get('grid_column'),
                'Core-ring family': family,
                'family_count': family_row['family_count'],
                'core_atom_count': len(record['core_atoms']),
                'substituent_atom_count': len(record['subs_atoms']),
                'SMILES': record['smiles'],
            }
            row.update(data.get(record['id'], {}))
            map_rows.append(row)

    map_df = pd.DataFrame(map_rows).sort_values(['family_position_index', 'Refcode'])
    return map_df, family_rows, selected_records, embedding


## Interactive HTML Rendering

In [ ]:
def format_value(value, digits=2, unit=None):
    if value in (None, '', 'None') or pd.isna(value):
        return '-'
    try:
        formatted = f'{float(value):.{digits}f}'
    except (TypeError, ValueError):
        formatted = str(value)
    return formatted if unit in (None, '') else f'{formatted} {unit}'

def mol_to_svg(mol, size=(260, 180)):
    if mol is None:
        return '<div class="missing-mol">Molecule unavailable</div>'
    svg = Draw.MolsToGridImage([mol], molsPerRow=1, subImgSize=size, useSVG=True)
    return svg.data if hasattr(svg, 'data') else str(svg)

def normalize_property_specs(properties_to_show):
    specs = []
    for label, spec in properties_to_show.items():
        if isinstance(spec, str):
            specs.append({'label': label, 'key': spec, 'unit': None, 'digits': 2})
        else:
            specs.append({
                'label': label,
                'key': spec.get('key', label),
                'unit': spec.get('unit'),
                'digits': spec.get('digits', 2),
            })
    return specs


def resolve_property_key(property_name, property_specs, available_columns):
    available_columns = list(available_columns)
    if property_name in available_columns:
        return property_name

    candidates = [property_name]
    for spec in property_specs:
        if property_name in (spec['label'], spec['key']):
            candidates.extend([spec['key'], spec['label']])
            if spec.get('unit'):
                candidates.extend([
                    f"{spec['key']} [{spec['unit']}]",
                    f"{spec['label']} [{spec['unit']}]",
                ])

    for candidate in candidates:
        if candidate in available_columns:
            return candidate

    def compact(text):
        return ''.join(char.lower() for char in str(text) if char.isalnum())

    compact_candidates = [compact(candidate) for candidate in candidates]
    for column in available_columns:
        compact_column = compact(column)
        if compact_column in compact_candidates or any(compact_column.startswith(candidate) for candidate in compact_candidates):
            return column

    return property_name

def resolve_property_specs(property_specs, available_columns):
    resolved_specs = []
    for spec in property_specs:
        resolved = dict(spec)
        resolved['key'] = resolve_property_key(spec['key'], property_specs, available_columns)
        resolved_specs.append(resolved)
    return resolved_specs

def is_numeric_series(values):
    converted = pd.to_numeric(pd.Series(values), errors='coerce')
    return converted.notna().sum() > 0 and converted.notna().sum() == pd.Series(values).notna().sum()


def numeric_colorbar_ticks(values, n_ticks=5, include_value=None, use_unique_when_few=False, calendar_years=False):
    values = pd.to_numeric(pd.Series(values), errors='coerce').dropna()
    if values.empty:
        return []

    n_ticks = max(2, int(n_ticks or 5))
    unique_values = np.array(sorted(values.unique()), dtype=float)
    integer_like = bool(np.all(np.isclose(unique_values, np.round(unique_values))))

    if calendar_years and unique_values.max() > unique_values.min():
        raw_step = (unique_values.max() - unique_values.min()) / max(n_ticks - 1, 1)
        magnitude = 10 ** np.floor(np.log10(max(raw_step, 1)))
        candidate_steps = magnitude * np.array([1, 2, 5, 10], dtype=float)
        step = candidate_steps[np.argmin(np.abs(candidate_steps - raw_step))]
        first_tick = np.floor(unique_values.min() / step) * step
        last_tick = np.ceil(unique_values.max() / step) * step
        tick_values = np.arange(first_tick, last_tick + 0.5 * step, step)
    elif use_unique_when_few and len(unique_values) <= n_ticks:
        tick_values = unique_values
    else:
        locator = MaxNLocator(nbins=max(n_ticks - 1, 1), integer=integer_like, min_n_ticks=2)
        tick_values = np.asarray(locator.tick_values(float(unique_values.min()), float(unique_values.max())), dtype=float)

    vmin = float(unique_values.min())
    vmax = float(unique_values.max())
    if include_value is not None and vmin <= float(include_value) <= vmax:
        tick_values = np.unique(np.append(tick_values, float(include_value)))
        tick_values = np.array(sorted(tick_values), dtype=float)

    def tick_label(value):
        if np.isclose(value, round(value)):
            return str(int(round(value)))
        return f'{value:.2f}'.rstrip('0').rstrip('.')

    return [
        {
            'value': tick_label(value),
            'numeric_value': float(value),
            'position': float((float(value) - vmin) / (vmax - vmin)) if vmax != vmin else 0.5,
        }
        for value in tick_values
    ]

def compact_property_name(color_property):
    return ''.join(char.lower() for char in str(color_property) if char.isalnum())

def is_t12_property(color_property):
    compact_name = compact_property_name(color_property)
    return compact_name in {'t12', 't12k', 't1over2', 't1slash2'}

def is_dhelec_property(color_property):
    return compact_property_name(color_property).startswith('dhelec')

def is_year_property(color_property):
    return compact_property_name(color_property) in {'year', 'publicationyear'}

def numeric_color_limits(color_property, vmin, vmax):
    if is_dhelec_property(color_property):
        return vmin, min(vmax, 80.0)
    return vmin, vmax

def numeric_colormap_name(color_property):
    return 'blue_white_red' if is_t12_property(color_property) or is_dhelec_property(color_property) else 'turbo'

def blue_white_red_colormap(center_position=0.5):
    center_position = float(np.clip(center_position, 0.0, 1.0))
    transition = min(0.04, center_position, 1.0 - center_position)
    return LinearSegmentedColormap.from_list(
        'blue_white_red',
        [
            (0.00, '#08306b'),  # dark blue, low T1/2
            (center_position - transition, '#deebf7'),
            (center_position, '#f7f7f7'),
            (center_position + transition, '#fee0d2'),
            (1.00, '#67000d'),  # dark red, high T1/2
        ],
    )

def numeric_colormap_and_norm(color_property, vmin, vmax):
    if is_t12_property(color_property) or is_dhelec_property(color_property):
        center = 298.0 if is_t12_property(color_property) else 0.0
        if vmin < center < vmax:
            center_position = (center - vmin) / (vmax - vmin)
            norm = Normalize(vmin=vmin, vmax=vmax)
            return blue_white_red_colormap(center_position), norm
        if vmin >= center:
            cmap = LinearSegmentedColormap.from_list('white_red', ['#f7f7f7', '#67000d'])
        else:
            cmap = LinearSegmentedColormap.from_list('blue_white', ['#08306b', '#f7f7f7'])
        norm = Normalize(vmin=vmin, vmax=vmax) if vmax != vmin else Normalize(vmin=0, vmax=1)
        return cmap, norm

    cmap = colormaps.get_cmap('turbo')
    norm = Normalize(vmin=vmin, vmax=vmax) if vmax != vmin else Normalize(vmin=0, vmax=1)
    return cmap, norm

def add_color_column(map_df, color_property, top_n_categories=20, cmap_name='tab20'):
    map_df = map_df.copy()
    values = map_df[color_property] if color_property in map_df.columns else pd.Series(['unknown'] * len(map_df))

    if is_numeric_series(values):
        numeric = pd.to_numeric(values, errors='coerce')
        valid_numeric = numeric.dropna()
        cmap_name_numeric = numeric_colormap_name(color_property)
        if valid_numeric.empty:
            map_df['_color_group'] = 'missing'
            map_df['_point_color'] = '#d9d9d9'
            legend_items = [{'kind': 'category', 'label': 'missing', 'color': '#d9d9d9'}]
            return map_df, legend_items

        vmin = float(valid_numeric.min())
        vmax = float(valid_numeric.max())
        color_vmin, color_vmax = numeric_color_limits(color_property, vmin, vmax)
        tick_payload = numeric_colorbar_ticks(
            [color_vmin, color_vmax],
            n_ticks=top_n_categories,
            calendar_years=is_year_property(color_property),
        )
        if is_dhelec_property(color_property):
            tick_payload = [tick for tick in tick_payload if tick['numeric_value'] <= color_vmax]
        if tick_payload:
            color_vmin = min(color_vmin, tick_payload[0]['numeric_value'])
            color_vmax = max(color_vmax, tick_payload[-1]['numeric_value'])
        cmap, norm = numeric_colormap_and_norm(color_property, color_vmin, color_vmax)
        map_df['_color_group'] = values.astype(str)
        missing_color = '#000000' if is_t12_property(color_property) else '#d9d9d9'
        map_df['_point_color'] = [to_hex(cmap(norm(np.clip(value, color_vmin, color_vmax)))) if pd.notna(value) else missing_color for value in numeric]
        for tick in tick_payload:
            tick['position'] = float(norm(tick['numeric_value'])) if color_vmax != color_vmin else 0.5
        legend_items = [{
            'kind': 'colorbar',
            'label': color_property,
            'cmap_name': cmap_name_numeric,
            'vmin': format_value(color_vmin),
            'vmax': format_value(color_vmax),
            'color_vmin': color_vmin,
            'color_vmax': color_vmax,
            'ticks': tick_payload,
            'colors': [to_hex(cmap(x)) for x in np.linspace(0, 1, 24)],
        }]
        return map_df, legend_items

    clean_values = values.fillna('unknown').astype(str)
    counts = clean_values.value_counts()
    top_values = list(counts.head(top_n_categories).index)
    groups = np.where(clean_values.isin(top_values), clean_values, 'Other')
    palette = colormaps.get_cmap(cmap_name)(np.linspace(0, 1, max(len(top_values), 1)))
    colors = {value: to_hex(palette[i % len(palette)]) for i, value in enumerate(top_values)}
    colors['Other'] = '#d9d9d9'
    map_df['_color_group'] = groups
    map_df['_point_color'] = [colors[group] for group in groups]
    order = top_values + (['Other'] if 'Other' in set(groups) else [])
    legend_items = [{'kind': 'category', 'label': f'{value} (n={(groups == value).sum()})', 'color': colors[value]} for value in order]
    return map_df, legend_items

def write_interactive_html(
    map_df,
    family_rows,
    descriptor_records,
    output_html_path,
    properties_to_show,
    color_property='last_author',
    title=None,
    caption=None,
    top_n_categories=20,
    plot_width=980,
    plot_height=680,
    iframe_height=760,
):
    output_html_path = Path(output_html_path)
    output_html_path.parent.mkdir(parents=True, exist_ok=True)
    property_specs = resolve_property_specs(normalize_property_specs(properties_to_show), map_df.columns)
    color_key = resolve_property_key(color_property, property_specs, map_df.columns)
    map_df, legend_items = add_color_column(map_df, color_key, top_n_categories=top_n_categories)

    margin = {'left': 42, 'right': 230, 'top': 18, 'bottom': 38}
    x_values = np.concatenate([map_df['map_x'].to_numpy(), np.array([row['family_x'] for row in family_rows])])
    y_values = np.concatenate([map_df['map_y'].to_numpy(), np.array([row['family_y'] for row in family_rows])])
    x_pad = 0.025 * (x_values.max() - x_values.min() or 1.0)
    y_pad = 0.025 * (y_values.max() - y_values.min() or 1.0)
    x_min, x_max = x_values.min() - x_pad, x_values.max() + x_pad
    y_min, y_max = y_values.min() - y_pad, y_values.max() + y_pad

    def scale_x(x):
        return margin['left'] + (x - x_min) / (x_max - x_min) * (plot_width - margin['left'] - margin['right'])

    def scale_y(y):
        return plot_height - margin['bottom'] - (y - y_min) / (y_max - y_min) * (plot_height - margin['top'] - margin['bottom'])

    ligand_mols_by_refcode = {record['id']: record.get('lig_mol', record.get('mol')) for record in descriptor_records}

    family_payload = []
    for row in family_rows:
        family_payload.append({
            'family_id': row['family_id'],
            'family': row['Core-ring family'],
            'family_count': int(row['family_count']),
            'x': float(scale_x(row['family_x'])),
            'y': float(scale_y(row['family_y'])),
        })

    point_payload = []
    for _, row in map_df.iterrows():
        mol_svg = mol_to_svg(ligand_mols_by_refcode.get(row['Refcode']))
        property_rows = []
        for spec in property_specs:
            value = row.get(spec['key'], None)
            property_rows.append(
                f"<tr><td>{escape(spec['label'])}</td><td>{escape(format_value(value, digits=spec['digits'], unit=spec['unit']))}</td></tr>"
            )
        property_html = '\n'.join(property_rows)
        tooltip_html = f"""
            <div class="tooltip-title">{escape(row['Refcode'])}</div>
            <div class="tooltip-subtitle">{escape(str(color_property))}: {escape(format_value(row.get(color_key, '-')))}</div>
            <div class="mol-svg">{mol_svg}</div>
            <table>
                <tr><td>Family</td><td>{escape(row['family_id'])} ({escape(row['Core-ring family'])})</td></tr>
                <tr><td>Core atoms</td><td>{int(row['core_atom_count'])}</td></tr>
                {property_html}
            </table>
        """
        point_payload.append({
            'refcode': row['Refcode'],
            'family_id': row['family_id'],
            'x': float(scale_x(row['map_x'])),
            'y': float(scale_y(row['map_y'])),
            'family_x': float(scale_x(row['family_x'])),
            'family_y': float(scale_y(row['family_y'])),
            'color': row['_point_color'],
            'tooltip_html': tooltip_html,
        })

    legend_payload = []
    for idx, item in enumerate(legend_items):
        payload = dict(item)
        payload['x'] = plot_width - margin['right'] + 30
        payload['y'] = margin['top'] + 25 + idx * 22
        legend_payload.append(payload)

    title = title or 'Interactive core-ring ligand map'
    caption = caption or 'Grey radial lines connect each compound to its family center. Hover points to inspect ligand and metadata.'

    html_doc = f"""<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>{escape(title)}</title>
<style>
body {{ margin: 0; font-family: Helvetica, Arial, sans-serif; background: #ffffff; color: #111827; }}
.wrapper {{ position: relative; width: {plot_width}px; margin: 0 auto; }}
.title {{ font-size: 18px; font-weight: 700; margin: 14px 0 2px 0; }}
.caption {{ font-size: 12px; color: #4b5563; margin-bottom: 8px; max-width: 820px; }}
svg {{ border: 1.4px solid black; background: white; }}
.branch-line {{ stroke: #c7c7c7; stroke-width: 0.8; pointer-events: none; }}
.family-center {{ fill: #111827; stroke: white; stroke-width: 1.2; opacity: 0.85; }}
.point {{ cursor: pointer; stroke: black; stroke-width: 0.45; opacity: 0.92; }}
.point:hover {{ stroke-width: 1.8; opacity: 1; }}
.axis-label {{ font-size: 13px; fill: #111827; }}
.legend-title {{ font-size: 12px; font-weight: 700; }}
.legend-unit {{ font-weight: 400; }}
.legend-text {{ font-size: 11px; dominant-baseline: middle; }}
.tooltip {{ position: absolute; display: none; width: 330px; max-height: 560px; overflow: auto; background: rgba(255,255,255,0.98); border: 1px solid #111827; box-shadow: 0 10px 30px rgba(0,0,0,0.18); border-radius: 8px; padding: 10px 12px; pointer-events: none; z-index: 10; }}
.tooltip-title {{ font-size: 15px; font-weight: 800; margin-bottom: 2px; }}
.tooltip-subtitle {{ font-size: 12px; color: #374151; margin-bottom: 6px; }}
.mol-svg svg {{ border: 0; display: block; margin: 0 auto 4px auto; }}
.missing-mol {{ height: 80px; display: flex; align-items: center; justify-content: center; color: #6b7280; }}
table {{ border-collapse: collapse; width: 100%; font-size: 11px; }}
td {{ padding: 2px 4px; border-top: 1px solid #e5e7eb; vertical-align: top; }}
td:first-child {{ color: #4b5563; width: 35%; }}
</style>
</head>
<body>
<div class="wrapper">
  <div class="title">{escape(title)}</div>
  <div class="caption">{escape(caption)}</div>
  <svg id="plot" width="{plot_width}" height="{plot_height}" viewBox="0 0 {plot_width} {plot_height}"></svg>
  <div id="tooltip" class="tooltip"></div>
</div>
<script>
const points = {json.dumps(point_payload)};
const families = {json.dumps(family_payload)};
const legend = {json.dumps(legend_payload)};
const legendTitle = {json.dumps(html_property_label_payload(color_property))};
const svg = document.getElementById('plot');
const tooltip = document.getElementById('tooltip');
const wrapper = document.querySelector('.wrapper');
const ns = 'http://www.w3.org/2000/svg';
function add(tag, attrs, text) {{
  const element = document.createElementNS(ns, tag);
  for (const [key, value] of Object.entries(attrs || {{}})) element.setAttribute(key, value);
  if (text !== undefined) element.textContent = text;
  svg.appendChild(element);
  return element;
}}
for (const point of points) {{ add('line', {{x1: point.family_x, y1: point.family_y, x2: point.x, y2: point.y, class: 'branch-line'}}); }}
for (const family of families) {{
  const center = add('circle', {{cx: family.x, cy: family.y, r: 4.4, class: 'family-center'}});
  const centerTitle = document.createElementNS(ns, 'title');
  centerTitle.textContent = family.family_id + ' | n=' + family.family_count + ' | ' + family.family;
  center.appendChild(centerTitle);
}}
function positionTooltip(event) {{
  const wrapperRect = wrapper.getBoundingClientRect();
  const localX = event.clientX - wrapperRect.left;
  const localY = event.clientY - wrapperRect.top;
  const tooltipWidth = tooltip.offsetWidth || 330;
  const tooltipHeight = tooltip.offsetHeight || 360;
  let left = localX + 18;
  let top = localY + 18;
  if (left + tooltipWidth > wrapperRect.width - 8) left = localX - tooltipWidth - 18;
  if (top + tooltipHeight > wrapperRect.height - 8) top = localY - tooltipHeight - 18;
  tooltip.style.left = Math.max(8, left) + 'px';
  tooltip.style.top = Math.max(8, top) + 'px';
}}
for (const point of points) {{
  const circle = add('circle', {{cx: point.x, cy: point.y, r: 5.2, fill: point.color, class: 'point'}});
  circle.addEventListener('mousemove', (event) => {{ tooltip.style.display = 'block'; tooltip.innerHTML = point.tooltip_html; requestAnimationFrame(() => positionTooltip(event)); }});
  circle.addEventListener('mouseleave', () => {{ tooltip.style.display = 'none'; }});
}}
<!-- add('text', {{x: {margin['left'] + (plot_width - margin['left'] - margin['right']) / 2}, y: {plot_height - 16}, class: 'axis-label', 'text-anchor': 'middle'}}, 'Map coordinate 1') -->;
<!-- add('text', {{x: 18, y: {margin['top'] + (plot_height - margin['top'] - margin['bottom']) / 2}, class: 'axis-label', transform: 'rotate(-90 18 {margin['top'] + (plot_height - margin['top'] - margin['bottom']) / 2})', 'text-anchor': 'middle'}}, 'Map coordinate 2') -->;
const legendTitleElement = add('text', {{x: {plot_width - margin['right'] + 30}, y: {margin['top']}, class: 'legend-title'}});
if (legendTitle.kind === 't12') {{
  legendTitleElement.appendChild(document.createTextNode('T'));
  const sub = document.createElementNS(ns, 'tspan');
  sub.setAttribute('baseline-shift', 'sub');
  sub.setAttribute('font-size', '75%');
  sub.textContent = '1/2';
  legendTitleElement.appendChild(sub);
  const unit = document.createElementNS(ns, 'tspan');
  unit.setAttribute('class', 'legend-unit');
  unit.textContent = ' [K]';
  legendTitleElement.appendChild(unit);
}} else if (legendTitle.kind === 'dhelec') {{
  legendTitleElement.appendChild(document.createTextNode('ΔH'));
  const sub = document.createElementNS(ns, 'tspan');
  sub.setAttribute('baseline-shift', 'sub');
  sub.setAttribute('font-size', '75%');
  sub.textContent = 'elec';
  legendTitleElement.appendChild(sub);
  const unit = document.createElementNS(ns, 'tspan');
  unit.setAttribute('class', 'legend-unit');
  unit.textContent = ' [kJ/mol]';
  legendTitleElement.appendChild(unit);
}} else {{
  legendTitleElement.textContent = legendTitle.text;
}}
for (const item of legend) {{
  if (item.kind === 'colorbar') {{
    const defs = svg.querySelector('defs') || add('defs', {{}});
    const gradientId = 'colorbar-gradient-' + Math.random().toString(36).slice(2);
    const gradient = document.createElementNS(ns, 'linearGradient');
    const barWidth = 22;
    const barHeight = 260;
    const barX = item.x;
    const barY = item.y + 8;
    gradient.setAttribute('id', gradientId);
    gradient.setAttribute('x1', '0%');
    gradient.setAttribute('x2', '0%');
    gradient.setAttribute('y1', '100%');
    gradient.setAttribute('y2', '0%');
    item.colors.forEach((color, idx) => {{
      const stop = document.createElementNS(ns, 'stop');
      stop.setAttribute('offset', (100 * idx / Math.max(item.colors.length - 1, 1)) + '%');
      stop.setAttribute('stop-color', color);
      gradient.appendChild(stop);
    }});
    defs.appendChild(gradient);
    add('rect', {{x: barX, y: barY, width: barWidth, height: barHeight, fill: 'url(#' + gradientId + ')', stroke: 'black', 'stroke-width': 0.6}});
    for (const tick of item.ticks) {{
      const tickY = barY + barHeight * (1 - tick.position);
      add('line', {{x1: barX + barWidth, y1: tickY, x2: barX + barWidth + 7, y2: tickY, stroke: 'black', 'stroke-width': 0.7}});
      add('text', {{x: barX + barWidth + 11, y: tickY + 3.5, class: 'legend-text'}}, tick.value);
    }}
  }} else {{
    add('circle', {{cx: item.x, cy: item.y, r: 5, fill: item.color, stroke: 'black', 'stroke-width': 0.4}});
    add('text', {{x: item.x + 11, y: item.y, class: 'legend-text'}}, item.label);
  }}
}}
</script>
</body>
</html>"""
    output_html_path.write_text(html_doc)
    return output_html_path




def pretty_property_label(label):
    if is_t12_property(label):
        return r'$\mathbf{T}_{\mathbf{1/2}}$ [K]'
    if is_dhelec_property(label):
        return r'$\mathbf{\Delta H}_{\mathbf{elec}}$ [kJ/mol]'
    return str(label).replace('_', ' ').strip().title()

def html_property_label_payload(label):
    if is_t12_property(label):
        return {'kind': 't12'}
    if is_dhelec_property(label):
        return {'kind': 'dhelec'}
    return {'kind': 'text', 'text': pretty_property_label(label)}

def write_static_png(
    map_df,
    family_rows,
    output_png_path,
    properties_to_show,
    color_property='last_author',
    top_n_categories=20,
    figsize=(7.9, 6.6),
    dpi=400,
):
    output_png_path = Path(output_png_path)
    output_png_path.parent.mkdir(parents=True, exist_ok=True)

    property_specs = resolve_property_specs(normalize_property_specs(properties_to_show), map_df.columns)
    color_key = resolve_property_key(color_property, property_specs, map_df.columns)
    plot_df, legend_items = add_color_column(map_df, color_key, top_n_categories=top_n_categories)

    fig, ax = plt.subplots(figsize=figsize)
    fig.subplots_adjust(left=0.025, right=0.825, top=0.995, bottom=0.005)

    for _, row in plot_df.iterrows():
        ax.plot(
            [row['family_x'], row['map_x']],
            [row['family_y'], row['map_y']],
            color='#c7c7c7',
            linewidth=0.55,
            zorder=1,
        )

    if family_rows:
        ax.scatter(
            [row['family_x'] for row in family_rows],
            [row['family_y'] for row in family_rows],
            s=20,
            color='#111827',
            edgecolor='white',
            linewidth=0.5,
            zorder=2,
        )

    ax.scatter(
        plot_df['map_x'],
        plot_df['map_y'],
        s=38,
        c=plot_df['_point_color'],
        edgecolor='black',
        linewidth=0.35,
        alpha=0.94,
        zorder=3,
    )

    x_values = np.concatenate([plot_df['map_x'].to_numpy(), np.array([row['family_x'] for row in family_rows])])
    y_values = np.concatenate([plot_df['map_y'].to_numpy(), np.array([row['family_y'] for row in family_rows])])
    x_pad = 0.025 * (x_values.max() - x_values.min() or 1.0)
    y_pad = 0.10 * (y_values.max() - y_values.min() or 1.0)
    ax.set_xlim(x_values.min() - x_pad, x_values.max() + x_pad)
    ax.set_ylim(y_values.min() - y_pad, y_values.max() + y_pad)
    ax.set_aspect('equal', adjustable='box')

    grid_families = [row for row in family_rows if row.get('grid_row') is not None and row.get('grid_column') is not None]
    if grid_families:
        col_positions = sorted({int(row['grid_column']): float(row['family_x']) for row in grid_families}.items())
        row_positions = sorted({int(row['grid_row']): float(row['family_y']) for row in grid_families}.items())
        ax.set_xticks([pos for _, pos in col_positions])
        ax.set_xticklabels([str(idx) for idx, _ in col_positions], fontsize=10)
        ax.set_yticks([pos for _, pos in row_positions])
        ax.set_yticklabels([str(idx) for idx, _ in row_positions], fontsize=10)
        ax.set_xlabel('Column', fontsize=11)
        ax.set_ylabel('Row', fontsize=11)
        ax.tick_params(axis='both', length=3.5, width=0.9, direction='out')
    else:
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xlabel('')
        ax.set_ylabel('')

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.3)
        spine.set_color('black')

    if legend_items and legend_items[0].get('kind') == 'colorbar':
        numeric = pd.to_numeric(plot_df[color_key], errors='coerce').dropna()
        if not numeric.empty:
            vmin = float(legend_items[0].get('color_vmin', numeric.min()))
            vmax = float(legend_items[0].get('color_vmax', numeric.max()))
            cmap_name_numeric = legend_items[0].get('cmap_name', numeric_colormap_name(color_key))
            cmap, norm = numeric_colormap_and_norm(color_key, vmin, vmax)
            scalar_mappable = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
            scalar_mappable.set_array([])
            fig.canvas.draw_idle()
            ax_box = ax.get_position()
            cbar_height = 0.84 * ax_box.height
            cbar_y = ax_box.y0 + 0.08 * ax_box.height
            cbar_ax = fig.add_axes([ax_box.x1 + 0.025, cbar_y, 0.026, cbar_height])
            cbar = fig.colorbar(scalar_mappable, cax=cbar_ax)
            cbar.set_label(pretty_property_label(color_property), fontsize=12)
            tick_payload = legend_items[0].get('ticks', [])
            if tick_payload:
                cbar.set_ticks([tick['numeric_value'] for tick in tick_payload])
                cbar.set_ticklabels([tick['value'] for tick in tick_payload])
            else:
                cbar.set_ticks(np.linspace(vmin, vmax, min(max(int(top_n_categories), 2), 8)) if vmax != vmin else [vmin])
            cbar.ax.tick_params(labelsize=11, length=4, width=0.9)
            cbar.outline.set_linewidth(1.0)
    elif legend_items:
        handles = [
            plt.Line2D(
                [0], [0],
                marker='o',
                linestyle='none',
                markerfacecolor=item['color'],
                markeredgecolor='black',
                markeredgewidth=0.4,
                markersize=7.0,
                label=item['label'],
            )
            for item in legend_items
        ]
        legend = ax.legend(
            handles=handles,
            title=pretty_property_label(color_property),
            loc='center left',
            bbox_to_anchor=(1.03, 0.5),
            frameon=False,
            fontsize=11,
            title_fontsize=12,
            borderaxespad=0.0,
        )
        legend.get_title().set_fontweight('bold')

    fig.savefig(output_png_path, dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    return output_png_path

def generate_core_ring_interactive_map(
    descriptor_records,        # Precomputed ligand descriptor dictionaries from collect_ligands_data().
    data,                      # Metadata keyed by refcode; values are added to the map table/tooltips.
    properties_to_show,        # Tooltip specification: display label -> metadata key/unit/digits.
    color_property,            # Column/property used to color the points and build the legend.
    target_denticity,          # Only ligands with this denticity are included in the map.
    output_path,               # Shared output basename; .html, .csv, and manuscript .png are created.
    embedding_method='mds',    # Method for placing families/members by fingerprint similarity: 'mds' or 'pca'.
    grid=False,                # If True, snap family centers to a sparse grid; otherwise keep continuous layout.
    top_n_categories=20,       # Number of categorical color groups kept before assigning the rest to 'Other'.
    grid_cell_spacing=1.0,     # Distance between family bins when grid=True.
    grid_member_radius=0.50,   # Maximum radius occupied by compounds around each family bin when grid=True.
    local_member_min_distance=0.08, # Minimum visual separation between compounds inside the same family, relative to local scale.
    spread_family_centers=True,# If True, gently separate continuous family centers to reduce overlap.
    family_min_distance=0.12,  # Minimum relative separation enforced between family centers in continuous maps.
    display_iframe=False,      # If True, display the generated HTML in the notebook immediately.
    png_figsize=(7.9, 6.6),    # Manuscript PNG size in inches; legend is placed outside the map frame.
    png_dpi=400,               # Resolution of the automatically generated PNG.
):
    map_df, family_rows, selected_records, embedding = build_core_ring_map_data(
        descriptor_records,
        target_denticity=target_denticity,
        data=data,
        embedding_method=embedding_method,
        grid=grid,
        spread_family_centers=spread_family_centers,
        family_min_distance=family_min_distance,
        grid_cell_spacing=grid_cell_spacing,
        grid_member_radius=grid_member_radius,
        local_member_min_distance=local_member_min_distance,
    )

    html_path = str(output_path)+'.html'
    png_path = str(output_path)+'.png'
    csv_path = str(output_path)+'.csv'

    # Creates .csv
    csv_columns = [
        'Refcode',
        'family_id',
        'family_count',
        'grid_row',
        'grid_column',
        'Core-ring family',
    ]
    property_specs = resolve_property_specs(normalize_property_specs(properties_to_show), map_df.columns)
    property_columns = [spec['key'] for spec in property_specs if spec['key'] in map_df.columns]
    csv_columns = [col for col in csv_columns + property_columns if col in map_df.columns]
    csv_columns = list(dict.fromkeys(csv_columns))
    map_df[csv_columns].to_csv(csv_path, index=False)

    layout_name = 'grid' if grid else 'tree'
    title = f'Denticity {target_denticity} homoleptic ligands: {layout_name} map'
    if grid: 
        caption = (
        f'Families are ordered using the {embedding_method.upper()} embedding method, then snapped onto a sparse grid.'
        ' This means that neighboring families should be more similar than far-away families.'
        ' Grey radial lines connect each compound to its family bin center; local positions use fingerprint similarity.'
        )
    else:
        caption = (f'Coordination-core ring families are positioned by fingerprint similarity using {embedding_method.upper()}, then spread for readability.'
        ' Grey radial lines connect each compound to its family center; local positions use fingerprint similarity.'
        )

    # Creates .html
    write_interactive_html(
        map_df,
        family_rows,
        selected_records,
        output_html_path=html_path,
        properties_to_show=properties_to_show,
        color_property=color_property,
        title=title,
        caption=caption,
        top_n_categories=top_n_categories,
    )

    # Creates manuscript-style .png without title, caption, or interactive elements.
    write_static_png(
        map_df,
        family_rows,
        output_png_path=png_path,
        properties_to_show=properties_to_show,
        color_property=color_property,
        top_n_categories=top_n_categories,
        figsize=png_figsize,
        dpi=png_dpi,
    )


    if display_iframe:
        display(IFrame(str(html_path), width='100%', height=760))

    return {
        'map_df': map_df,
        'family_rows': family_rows,
        'descriptor_records': descriptor_records,
        'selected_records': selected_records,
        'html_path': html_path,
        'csv_path': csv_path,
        'png_path': png_path,
        'embedding': embedding,
    }


## Debug Helpers

In [11]:

def atom_debug_table(record):
    mol = record['lig_mol']
    core_atoms = set(record['core_atoms'])
    substituent_atoms = set(record['subs_atoms'])
    ring_membership = {idx: [] for idx in range(mol.GetNumAtoms())}
    for ring_id, ring in enumerate(mol.GetRingInfo().AtomRings()):
        for atom_idx in ring:
            ring_membership[atom_idx].append(ring_id)

    rows = []
    for atom in mol.GetAtoms():
        idx = atom.GetIdx()
        rows.append({
            'atom_idx': idx,
            'symbol': atom.GetSymbol(),
            'is_aromatic': atom.GetIsAromatic(),
            'ring_ids': ring_membership[idx],
            'core': idx in core_atoms,
            'substituent': idx in substituent_atoms,
            'neighbors': [nbr.GetIdx() for nbr in atom.GetNeighbors()],
        })
    return pd.DataFrame(rows)

def draw_core_assignment(record):
    mol = record['lig_mol']
    core_atoms = sorted(record['core_atoms'])
    substituent_ring_atoms = sorted(
        atom_idx for ring in mol.GetRingInfo().AtomRings() for atom_idx in ring
        if atom_idx not in set(core_atoms)
    )

    highlight_atoms = sorted(set(core_atoms) | set(substituent_ring_atoms))
    highlight_colors = {}
    for idx in substituent_ring_atoms:
        highlight_colors[idx] = (0.30, 0.55, 1.00)
    for idx in core_atoms:
        highlight_colors[idx] = (0.20, 0.75, 0.35)

    svg = Draw.MolsToGridImage(
        [mol],
        molsPerRow=1,
        subImgSize=(520, 320),
        legends=[f"{record['id']} | green=core atoms, blue=substituent ring atoms"],
        highlightAtomLists=[highlight_atoms],
        highlightAtomColors=[highlight_colors],
        useSVG=True,
    )
    display(SVG(svg.data if hasattr(svg, 'data') else str(svg)))

def summarize_core_families(records):
    return pd.DataFrame([
        {
            'Refcode': record['id'],
            'denticity': record['denticity'],
            'SMILES': record['smiles'],
            'Core-ring family': record['family'],
            'core_atom_count': len(record['core_atoms']),
            'substituent_atom_count': len(record['subs_atoms']),
        }
        for record in records
    ])
